## Imports de Librerías a utilizar

In [2]:
import seaborn as sns 
import matplotlib.pyplot as plt 
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, GridSearchCV

## Carga del Dataset de Triaje

In [3]:
# Carga de Triaje (Datos vitales sobre los pacientes)
triaje = pd.read_csv(
    "../datos/mimic-iv-ed-demo-2.2/mimic-iv-ed-demo-2.2/ed/triage.csv.gz"
)

# Carga de edstays (Datos sobre como llegan nuestros pacientes al hospital)
edstays = pd.read_csv(
    "../datos/mimic-iv-ed-demo-2.2/mimic-iv-ed-demo-2.2/ed/edstays.csv.gz",
    usecols=["stay_id", "intime", "arrival_transport"],
)

## Limpieza del Dataset

In [4]:
# Filtramos los valores nulos de 'acuity, temperature y heartrate' y eliminamos las columnas que contienen los IDs
df = triaje[triaje['acuity'].notna()].drop(columns=['subject_id', 'stay_id']).copy()

# Hacemos un merge de ambos Dataframes utilizando como clave 'stay_id', manteniendo todos los datos de 'triaje' con left join.
df = triaje[triaje["acuity"].notna()].merge(edstays, on="stay_id", how="left")

# Creamos la columna 'arrival_hour' a partir del valor intime que contiene la hora exacta de llegada, así no generamos ruido.
df["arrival_hour"] = pd.to_datetime(df["intime"]).dt.hour 

# Eliminamos las columnas que no nos proporcionan datos consistentes
df = df.drop(columns=["subject_id", "stay_id", "intime"])

# Obtenemos todos los valores de la columna 'pain' y cambiamos el typo que se ha encontrado por un 0.
pain_txt = df["pain"].astype("string").str.strip().str.lower().replace({"o": "0"})

# Guardamos la columna 'pain' en una variable pain_num, esta contendrá la columna 'pain' 
# construida en base a números y completamente normzaliada
pain_num = df["pain"] = pd.to_numeric(pain_txt, errors="coerce")


df["pain_not_assessable"] = (pain_txt.notna() & pain_num.isna()).astype(int)
df["pain"] = pain_num.where(pain_num.between(0, 10))

df

,temperature,heartrate,resprate,o2sat,sbp,dbp,pain,acuity,chiefcomplaint,arrival_transport,arrival_hour,pain_not_assessable
0,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,2.0,"Abd pain, LEAKING ASCITES",AMBULANCE,22,0
1,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,2.0,"Dizziness, Diplopia",AMBULANCE,14,1
2,98.8,72.0,18.0,90.0,98.0,48.0,<NA>,2.0,Abd pain,UNKNOWN,23,0
3,97.8,115.0,20.0,99.0,92.0,42.0,0,2.0,"DKA, Transfer",AMBULANCE,6,0
4,98.1,95.0,18.0,99.0,149.0,74.0,0,2.0,Psych eval,WALK IN,11,0
...,...,...,...,...,...,...,...,...,...,...,...,...
202,97.7,111.0,15.0,100.0,133.0,79.0,3,3.0,"Dehydration, Nausea, Rash",WALK IN,14,0
203,98.4,106.0,18.0,95.0,131.0,76.0,5,3.0,"Abd pain, Transfer",AMBULANCE,22,0
204,96.8,78.0,16.0,95.0,186.0,66.0,8,3.0,Abd pain,WALK IN,0,0
205,98.6,108.0,18.0,95.0,143.0,61.0,7,3.0,"Abd pain, Back pain",AMBULANCE,17,0


## Feature Engineering sobre los datos que ya tenemos, aplicando fórmulas ya establecidas

In [ ]:
# Hacemos Feature Engineering sobre los datos que ya tenemos filtrados

# Creamos un nuevo dato, el índice de shock, que supone la división de los latidos entre la presión arterial sistolica
df["shock_index"] = df["heartrate"] / df["sbp"]

# Obtenemos la presión del pulso a partir de la división de la presion arterial sistólica entre la diastólica
df["pulse_pressure"] = df["sbp"] - df["dbp"]

# Rellenamos los huecos vacíos con cadenas vacías, más adelante convertiremos esta columna en números para el modelo.
df["chiefcomplaint"] = df["chiefcomplaint"].fillna("")

## Preparación de Observaciones para el modelo y de las etiquetas

In [157]:
# Declaramos las columnas númericas 
numerical_columns = ["temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp", "pain",
                     "pain_not_assessable", "shock_index", "pulse_pressure", "arrival_hour"]

# Definimos las observaciones
X = df[numerical_columns + ["arrival_transport", "chiefcomplaint"]]

# Definimos las etiquetas o los valores que tiene que predecir el modelo (Hacemos que 1 y 2 se normaicen a 0 y 3,4,5 se normalicen a 1)
y = (df["acuity"] <= 2).astype(int)

## Preparación de Datos Númericos y Texto por separado

In [ ]:
# Declaramos la preparación que haremos sobre los datos
prep = make_column_transformer(
    # Con SimpleImputer rellenamos los huecos que no contengan números para no sesgar los datos
    (make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), numerical_columns),
    # Como en `arrival_transport` separamos en categorías, OneHotEncoder nos ayuda a separar las cateorías y transformarlas a números
    (OneHotEncoder(handle_unknown="ignore"), ["arrival_transport"]),
    # Normalizamos la columna de 'chiefcomplaint' estableciendo pesos según las repeticiones de las palabras
    (TfidfVectorizer(ngram_range=(1,2), min_df=2), "chiefcomplaint")
)

## Creación del Modelo 

In [ ]:
# Declaramos el modelo que vamos a entrenar utilizando los datos que hemos preparado en los pasos anteriores, además utilizamos el algoritmo
# de LogisticRegression con un máximo de 5000 iteraciones sobre nuestros datos
model = make_pipeline(prep, LogisticRegression(max_iter=5000))

# Usamos RepeatedStratifiedKFold para recorrer los datos de múltiples maneras distintas y evaluar el modelo con mayor fiabilidad.
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

# Utilizamos GridSearchCV para poder especificar el modelo que vamos a entrenar y los hiperparámetros que vamos a utilizar
grid = GridSearchCV(
    model,
    {
        "logisticregression__C": [0.01, 0.03, 0.1, 0.3, 1, 3, 10],
        "logisticregression__class_weight": ["balanced", None],
        "columntransformer__tfidfvectorizer__min_df": [1, 2, 3],
        # Especificamos las flags que vamos a usar a la hora de usar GridSearchCV
    }, cv=cv, scoring=["accuracy", "neg_log_loss"], n_jobs=-1, refit='neg_log_loss', verbose=1, return_train_score=True
)

## Entrenamiento del modelo

In [ ]:
# Entrenamiento del modelo
grid.fit(X,y)

Fitting 15 folds for each of 42 candidates, totalling 630 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step..._iter=5000))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'columntransformer__tfidfvectorizer__min_df': [1, 2, ...], 'logisticregression__C': [0.01, 0.03, ...], 'logisticregression__class_weight': ['balanced', None]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","['accuracy', 'neg_log_loss']"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'neg_log_loss'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22

## Visualización de las configuraciones que ha generado el modelo (De mayor a menor)

In [ ]:
# Creamos un Dataframe de pandas con todas las configuraciones que nos ha generado el entrenamiento
res = pd.DataFrame(grid.cv_results_)

configs_df = res[[
    "mean_test_accuracy",
    "mean_test_neg_log_loss",
    "param_logisticregression__C",
    "param_logisticregression__class_weight",
    "param_columntransformer__tfidfvectorizer__min_df"
]]

# Renombramos las columnas para una mejor legibilidad de los hiperparámetros utilizados en el entrenamiento
configs_df = configs_df.rename(columns={'logisticregression__C':'C', 
                                        'param_logisticregression__class_weight': 'class_weight',
                                        'param_columntransformer__tfidfvectorizer__min_df': 'min_df' })

# Transformamos la pérdida a positivo
configs_df['mean_test_neg_log_loss'] = -res['mean_test_neg_log_loss']

# Mostramos los valores ordenados de mayor a menor
configs_df.sort_values('mean_test_accuracy', ascending=False)[:10]

,mean_test_accuracy,mean_test_neg_log_loss,param_logisticregression__C,class_weight,min_df
39,0.705381,0.616348,3.00,NaN,3
25,0.702245,0.611404,3.00,NaN,2
9,0.702207,0.604196,1.00,NaN,1
37,0.700658,0.595383,1.00,NaN,3
23,0.700581,0.594130,1.00,NaN,2
41,0.697406,0.673157,10.00,NaN,3
35,0.694464,0.589992,0.30,NaN,3
11,0.694154,0.625343,3.00,NaN,1
21,0.692722,0.591258,0.30,NaN,2
38,0.690979,0.617411,3.00,balanced,3


## Visualización de las puntuaciones (Función de Pérdida y Precisión)

In [201]:
# Declaramos la combinación final que hemos elegido en base a la tabla de configuraciones qie ha generado el entrenamiento
final_params = {
    "columntransformer__tfidfvectorizer__min_df": 2,
    "logisticregression__C": 1,
    "logisticregression__class_weight": None,
}

# Obtenemos el índice de la combinación que queremos utilizar como defnitiva
i = grid.cv_results_['params'].index(final_params)

# Obtenemos precisión y pérdida del modelo.
accuracy = grid.cv_results_['mean_test_accuracy'][i]
loss = -grid.cv_results_['mean_test_neg_log_loss'][i]

# Los mostramos por pantalla.
print(f'Accuraccy: {accuracy}')
print(f'Loss: {loss}')

Accuraccy: 0.7005807200929152
Loss: 0.5941297158102204


## Conexión a MLFlow

In [ ]:
import mlflow, mlflow.sklearn, mlflow.tensorflow
import numpy as np, pandas as pd, tensorflow as tf

mlflow.set_tracking_uri("http://localhost:5000")
vitales = ['temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'pain']

## Pesos sobre la red neuronal de TensorFlow

In [ ]:
model = mlflow.tensorflow.load_model("models:/Clasificador_TF_Triaje/latest")

densa = [l for l in model.layers if isinstance(l, tf.keras.layers.Dense)][0]
W, b = densa.get_weights()   # W: (7 variables, nº de neuronas)

pesos_tf = pd.DataFrame(W, index=vitales)   # matriz completa de pesos
pesos_tf.abs().mean(axis=1).sort_values(ascending=False).rename("peso_medio_abs")

## Pesos sobre el Random Forest

In [ ]:
rf_triaje = mlflow.sklearn.load_model("models:/Clasificador_RF_Triaje/latest")

pd.Series(rf_triaje.feature_importances_, index=vitales).sort_values(ascending=False).rename("importancia")

## Pesos sobre la Regresión Logística

In [ ]:
pipe = mlflow.sklearn.load_model("models:/Clasificador_LR_Triaje/latest")

nombres = pipe[:-1].get_feature_names_out()
coefs = pd.Series(pipe[-1].coef_[0], index=nombres, name="coeficiente")

display(coefs[coefs.index.str.startswith("pipeline__")].sort_values(key=abs, ascending=False))   # variables numéricas
coefs.sort_values(key=abs, ascending=False).head(15)                                             # top 15 global